# Preprocessing & Feature Engineering

## Objective
1. Define the prediction task and prediction point precisely.
2. Perform a systematic leakage analysis on every feature.
3. Clean the data (handle missing / 'unknown' values, duplicates).
4. Engineer meaningful, non-leaky features.
5. Build a reproducible scikit-learn preprocessing pipeline.
6. Save the processed dataset for modelling.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42

df = pd.read_csv('../data/raw/bank_marketing.csv')
print(f"Loaded dataset: {df.shape}")
df.head()


Loaded dataset: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


## 1. Prediction Task Definition

**Business question:**  
> Can historical customer and campaign data predict whether a customer will subscribe to a term deposit?

**ML task:** Binary classification  
**Target variable:** `y` — whether the client subscribed to a term deposit (`yes` / `no`)  
**Positive class:** `yes` (subscribed)  
**Negative class:** `no` (did not subscribe)

### Prediction Point
The prediction is intended to be made **before** contacting a customer in a new campaign round.  
At that moment, the bank has access to:
- Customer profile data (collected from CRM / banking records)
- Results and history from **previous** campaigns
- Planned campaign logistics (contact type, month, day)

The bank does **not** yet have access to:
- The duration of the current call (`duration`) — only known after the call ends
- Any outcome of the current call


## 2. Leakage Analysis

For every feature, we ask: **would this information be available at prediction time (before contacting the customer)?**

In [2]:
leakage_table = pd.DataFrame([
    # Feature, Available at prediction time?, Leakage risk, Decision, Reason
    ("age",       "Yes — from CRM",                    "None",   "Keep",    "Stable customer attribute"),
    ("job",       "Yes — from CRM",                    "None",   "Keep",    "Stable customer attribute"),
    ("marital",   "Yes — from CRM",                    "None",   "Keep",    "Stable customer attribute"),
    ("education", "Yes — from CRM",                    "None",   "Keep",    "Stable customer attribute"),
    ("default",   "Yes — from CRM",                    "None",   "Keep",    "Stable customer attribute"),
    ("balance",   "Yes — from banking records",         "None",   "Keep",    "Available pre-contact"),
    ("housing",   "Yes — from banking records",         "None",   "Keep",    "Available pre-contact"),
    ("loan",      "Yes — from banking records",         "None",   "Keep",    "Available pre-contact"),
    ("contact",   "Partially — planned before call",   "Low",    "Keep",    "Contact type is decided before calling"),
    ("day_of_week","Yes — campaign schedule",           "None",   "Keep",    "Known when scheduling the call"),
    ("month",     "Yes — campaign schedule",            "None",   "Keep",    "Known when scheduling the call"),
    ("duration",  "NO — only known after the call",    "HIGH",   "DROP",    "UCI documentation explicitly flags this as leakage"),
    ("campaign",  "Yes — count of contacts so far",    "None",   "Keep",    "Count of contacts in current campaign"),
    ("pdays",     "Yes — from previous campaign data", "None",   "Keep",    "Days since last contact from previous campaign"),
    ("previous",  "Yes — from previous campaign data", "None",   "Keep",    "Number of previous contacts"),
    ("poutcome",  "Yes — from previous campaign data", "None",   "Keep",    "Outcome of the previous campaign"),
    ("y",         "N/A — this is the TARGET",          "N/A",    "Target",  "Binary classification target"),
], columns=["Feature", "Available at Prediction Time", "Leakage Risk", "Decision", "Reason"])

print(leakage_table.to_string(index=False))


    Feature      Available at Prediction Time Leakage Risk Decision                                             Reason
        age                    Yes — from CRM         None     Keep                          Stable customer attribute
        job                    Yes — from CRM         None     Keep                          Stable customer attribute
    marital                    Yes — from CRM         None     Keep                          Stable customer attribute
  education                    Yes — from CRM         None     Keep                          Stable customer attribute
    default                    Yes — from CRM         None     Keep                          Stable customer attribute
    balance        Yes — from banking records         None     Keep                              Available pre-contact
    housing        Yes — from banking records         None     Keep                              Available pre-contact
       loan        Yes — from banking records   

### Leakage Decision
| Feature | Decision | Reason |
|---------|----------|--------|
| `duration` | **DROP** | Only known after the call ends. Including it would artificially inflate model performance without being usable in a real deployment scenario. The UCI dataset authors explicitly warn against using it for realistic prediction models. |

All other features are available at the intended prediction point and will be retained.


## 3. Data Cleaning

In [3]:
# 3.1 Drop leakage column
df_clean = df.drop(columns=['duration'])
print(f"Shape after dropping 'duration': {df_clean.shape}")


Shape after dropping 'duration': (45211, 16)


In [4]:
# 3.2 Inspect 'unknown' values
cat_cols = df_clean.select_dtypes(include='object').columns.drop('y').tolist()
print("'unknown' counts and response rates:\n")
for col in cat_cols:
    n = (df_clean[col] == 'unknown').sum()
    if n > 0:
        r_unk  = (df_clean[df_clean[col] == 'unknown']['y'] == 'yes').mean() * 100
        r_knwn = (df_clean[df_clean[col] != 'unknown']['y'] == 'yes').mean() * 100
        print(f"  {col}: {n} unknowns ({n/len(df_clean)*100:.1f}%)")
        print(f"    Response rate — unknown: {r_unk:.1f}%  |  known: {r_knwn:.1f}%")


'unknown' counts and response rates:



### Treatment of 'unknown' values
We **retain** `'unknown'` as an explicit category for all columns.

**Reasoning:**
- `poutcome` has 81.7% unknowns — mass imputation would destroy this column.
- `contact` has 28.8% unknowns — the unknown contact type may itself be predictive.
- `education` and `job` have small unknown counts but different response rates from the known population.
- Keeping 'unknown' as a category is the safest, most transparent choice for tree-based models and one-hot encoders.


In [5]:
# 3.3 Encode target: yes → 1, no → 0
df_clean['y'] = (df_clean['y'] == 'yes').astype(int)
print("Target encoding: yes=1, no=0")
print(df_clean['y'].value_counts())


Target encoding: yes=1, no=0
y
0    39922
1     5289
Name: count, dtype: int64


In [6]:
# 3.4 Verify no exact duplicates
print(f"Duplicate rows: {df_clean.duplicated().sum()}")


Duplicate rows: 16


## 4. Feature Engineering

In [7]:
# 4.1 was_previously_contacted — binary flag from pdays
# pdays = -1 means client was never contacted in a previous campaign
df_clean['was_previously_contacted'] = (df_clean['pdays'] != -1).astype(int)
print("was_previously_contacted:")
print(df_clean.groupby('was_previously_contacted')['y'].mean().rename('response_rate').round(3))


was_previously_contacted:
was_previously_contacted
0    0.092
1    0.231
Name: response_rate, dtype: float64


In [8]:
# 4.2 age_group — binned age for interpretability
df_clean['age_group'] = pd.cut(
    df_clean['age'],
    bins=[0, 30, 40, 50, 60, 100],
    labels=['<30', '30-40', '40-50', '50-60', '60+']
)
print("Response rate by age group:")
print(df_clean.groupby('age_group', observed=True)['y'].mean().mul(100).round(1).rename('response_rate_%'))


Response rate by age group:
age_group
<30      16.3
30-40    10.2
40-50     9.1
50-60    10.1
60+      42.3
Name: response_rate_%, dtype: float64


In [9]:
# 4.3 campaign_intensity — cap extreme campaign contact counts
# Heavy outliers: some clients were contacted >50 times
print("campaign contacts — 95th percentile:", df_clean['campaign'].quantile(0.95))
df_clean['campaign_capped'] = df_clean['campaign'].clip(upper=int(df_clean['campaign'].quantile(0.95)))


campaign contacts — 95th percentile: 8.0


In [10]:
# Summary of engineered features
print("Engineered features added:")
print("  was_previously_contacted — binary flag (pdays != -1)")
print("  age_group                — binned age category")
print("  campaign_capped          — campaign contacts capped at 95th percentile")
print(f"\nDataset shape: {df_clean.shape}")


Engineered features added:
  was_previously_contacted — binary flag (pdays != -1)
  age_group                — binned age category
  campaign_capped          — campaign contacts capped at 95th percentile

Dataset shape: (45211, 19)


## 5. Train / Validation / Test Split

**Strategy:** 70% train | 15% validation | 15% test  
**Stratification:** Yes — preserves class ratio across all splits  
**Random state:** 42 (reproducibility)

The test set is held out completely and will only be used for final evaluation after all model and threshold decisions are made.


In [11]:
X = df_clean.drop(columns=['y'])
y = df_clean['y']

# First split: train (70%) vs temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

# Second split: validation (15%) vs test (15%) from temp
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Train:      {X_train.shape[0]:,} rows  | positive rate: {y_train.mean():.3f}")
print(f"Validation: {X_val.shape[0]:,} rows  | positive rate: {y_val.mean():.3f}")
print(f"Test:       {X_test.shape[0]:,} rows  | positive rate: {y_test.mean():.3f}")


Train:      31,647 rows  | positive rate: 0.117
Validation: 6,782 rows  | positive rate: 0.117
Test:       6,782 rows  | positive rate: 0.117


## 6. Preprocessing Pipeline

In [12]:
# Identify feature groups for the ColumnTransformer
numerical_features = [
    'age', 'balance', 'day_of_week', 'campaign', 'campaign_capped',
    'pdays', 'previous', 'was_previously_contacted'
]

ordinal_features = ['age_group']
ordinal_categories = [['<30', '30-40', '40-50', '50-60', '60+']]

categorical_features = [
    'job', 'marital', 'education', 'default',
    'housing', 'loan', 'contact', 'month', 'poutcome'
]

print("Numerical features: ", numerical_features)
print("Ordinal features:   ", ordinal_features)
print("Categorical features:", categorical_features)


Numerical features:  ['age', 'balance', 'day_of_week', 'campaign', 'campaign_capped', 'pdays', 'previous', 'was_previously_contacted']
Ordinal features:    ['age_group']
Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [13]:
# Build sub-pipelines
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # safety net — no actual NaNs expected
    ('scaler',  StandardScaler())
])

ordinal_pipeline = Pipeline([
    ('encoder', OrdinalEncoder(categories=ordinal_categories))
])

categorical_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline,   numerical_features),
    ('ord', ordinal_pipeline,     ordinal_features),
    ('cat', categorical_pipeline, categorical_features),
], remainder='drop')

print("Preprocessing pipeline constructed.")


Preprocessing pipeline constructed.


In [14]:
# Fit on training data only — transform all splits
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)

# Get feature names after encoding
cat_feature_names = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(categorical_features)
all_feature_names = numerical_features + ordinal_features + list(cat_feature_names)

print(f"Processed training shape:   {X_train_proc.shape}")
print(f"Processed validation shape: {X_val_proc.shape}")
print(f"Processed test shape:       {X_test_proc.shape}")
print(f"Total features after encoding: {X_train_proc.shape[1]}")


Processed training shape:   (31647, 53)
Processed validation shape: (6782, 53)
Processed test shape:       (6782, 53)
Total features after encoding: 53


In [15]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Save as DataFrames with column names
pd.DataFrame(X_train_proc, columns=all_feature_names).assign(y=y_train.values).to_csv('../data/processed/train.csv', index=False)
pd.DataFrame(X_val_proc,   columns=all_feature_names).assign(y=y_val.values).to_csv('../data/processed/val.csv',   index=False)
pd.DataFrame(X_test_proc,  columns=all_feature_names).assign(y=y_test.values).to_csv('../data/processed/test.csv', index=False)

# Also save the unprocessed splits (for non-pipeline models / analysis)
X_train.assign(y=y_train.values).to_csv('../data/processed/train_raw.csv', index=False)
X_val.assign(y=y_val.values).to_csv('../data/processed/val_raw.csv',       index=False)
X_test.assign(y=y_test.values).to_csv('../data/processed/test_raw.csv',    index=False)

print("Saved processed splits to data/processed/")


Saved processed splits to data/processed/


## Key Findings

| Item | Decision | Reason |
|------|----------|--------|
| Target | `y` binary (0/1) | Directly from dataset; yes→1, no→0 |
| Prediction point | Before current campaign contact | Only pre-contact features used |
| `duration` | **Dropped** | Target leakage — only known after call |
| `'unknown'` values | Retained as explicit category | Carries predictive signal; mass imputation unsupported |
| Split | 70/15/15 stratified | Preserves class imbalance across splits |
| New features | `was_previously_contacted`, `age_group`, `campaign_capped` | Non-leaky, interpretable, evidence-based |
| Scaling | StandardScaler on numerical | Required for Logistic Regression; harmless for trees |
| Encoding | OneHotEncoder for nominal, OrdinalEncoder for `age_group` | Appropriate for feature types |

## Decisions Made

- **`duration` excluded** — leakage. Documented in `reports/decision_log.md`.
- **`'unknown'` retained** — treated as a valid category by the OneHotEncoder.
- **Preprocessing pipeline fitted on training data only** — no data leaks from validation/test.
- **Test set locked** — will only be used after all model and threshold decisions are finalised.
